#### Description: real application for U.S. energy generation compositions and New York City Citi bike sharing system
#### generate required LSTM model results needed for Figures S5 and S11.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import copy
from scipy.special import rel_entr
import time
import random

plt.style.use('seaborn-v0_8-darkgrid')


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Force deterministic operations on GPU
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(1234)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
class SphericalLSTM(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=64, num_layers=1):
        super(SphericalLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, 
                            num_layers=num_layers, batch_first=True)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        
        # UPGRADE: 2-Layer MLP gives the network capacity to project to the sphere
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        normed_hidden = self.layer_norm(lstm_out[:, -1, :])
        raw_pred = self.fc(normed_hidden)
        return raw_pred / torch.norm(raw_pred, p=2, dim=1, keepdim=True)

class SphericalLSTM_WithTime(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=64, num_layers=1):
        super(SphericalLSTM_WithTime, self).__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, 
                            num_layers=num_layers, batch_first=True)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        
        # UPGRADE: 2-Layer MLP
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        normed_hidden = self.layer_norm(lstm_out[:, -1, :])
        raw_pred = self.fc(normed_hidden)
        return raw_pred / torch.norm(raw_pred, p=2, dim=1, keepdim=True)

In [ ]:
class SphericalForecaster:
    def __init__(self, model, seq_length=12, max_epochs=150, lr=0.005, patience=10, weight_decay=0.0):
        self.seq_length = seq_length
        self.max_epochs = max_epochs
        self.lr = lr
        self.patience = patience
        self.weight_decay = weight_decay
        self.model = model.to(device)
        
        # UPGRADE: Revert to MSE (Mathematically equivalent to Cosine for unit vectors, but better gradients)
        self.criterion = nn.MSELoss() 
        
    def create_sequences(self, X_data, Y_data):
        X_seq, y_seq = [], []
        for i in range(len(X_data) - self.seq_length):
            X_seq.append(X_data[i : i + self.seq_length])
            y_seq.append(Y_data[i + self.seq_length])
        return torch.tensor(np.array(X_seq), dtype=torch.float32), torch.tensor(np.array(y_seq), dtype=torch.float32)

    def fit(self, X_window, Y_window, val_ratio=0.2, batch_size=32):
        for layer in self.model.children():
            if hasattr(layer, 'reset_parameters'): layer.reset_parameters()
                
        X_all, y_all = self.create_sequences(X_window, Y_window)
        split_idx = int(len(X_all) * (1 - val_ratio))
        
        X_train, y_train = X_all[:split_idx].to(device), y_all[:split_idx].to(device)
        X_val, y_val = X_all[split_idx:].to(device), y_all[split_idx:].to(device)
        
        # UPGRADE: Mini-Batching via DataLoader
        train_dataset = TensorDataset(X_train, y_train)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        
        optimizer = optim.Adam(self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
        
        best_val_loss, patience_counter, best_weights = float('inf'), 0, None
        
        for epoch in range(self.max_epochs):
            self.model.train()
            
            for batch_x, batch_y in train_loader:
                optimizer.zero_grad()
                
                loss = self.criterion(self.model(batch_x), batch_y)
                loss.backward()
                optimizer.step()
            
            # Validation Phase
            self.model.eval()
            with torch.no_grad():
                val_loss = self.criterion(self.model(X_val), y_val).item()
                
            scheduler.step(val_loss)
                
            if val_loss < best_val_loss:
                best_val_loss, patience_counter = val_loss, 0
                best_weights = copy.deepcopy(self.model.state_dict())
            else:
                patience_counter += 1
                if patience_counter >= self.patience: break
                
        if best_weights is not None: self.model.load_state_dict(best_weights)

    def predict(self, initial_x_seq, steps_ahead, future_exo=None):
        self.model.eval()
        curr_seq = torch.tensor(initial_x_seq, dtype=torch.float32).unsqueeze(0).to(device)
        preds = []
        with torch.no_grad():
            for i in range(steps_ahead):
                next_y = self.model(curr_seq) 
                preds.append(next_y.cpu().numpy()[0])
                if future_exo is not None:
                    exo = torch.tensor(future_exo[i], dtype=torch.float32).to(device)
                    next_x = torch.cat((next_y[0], exo), dim=0).unsqueeze(0).unsqueeze(0)
                else:
                    next_x = next_y.unsqueeze(1)
                curr_seq = torch.cat((curr_seq[:, 1:, :], next_x), dim=1)
        return np.array(preds)

In [ ]:
def calculate_spherical_errors(predictions, targets):
    y_pred = predictions / np.linalg.norm(predictions, axis=1, keepdims=True)
    y_true = targets / np.linalg.norm(targets, axis=1, keepdims=True)
    dot_products = np.clip(np.sum(y_pred * y_true, axis=1), -1.0, 1.0)
    return np.mean(np.arccos(dot_products))

def calculate_js_divergence(predictions, targets):
    """
    Calculates the average Jensen-Shannon Divergence.
    Original and predicted data are squared first to form probability distributions.
    """
    # 1. Enforce strict L2 normalization (just in case model output drifted)
    y_pred = predictions / np.linalg.norm(predictions, axis=1, keepdims=True)
    y_true = targets / np.linalg.norm(targets, axis=1, keepdims=True)
    
    # 2. Square the data (Maps spherical coordinates to probability distributions)
    P = np.square(y_true)
    Q = np.square(y_pred)
    
    # Add a microscopic epsilon to prevent log(0) errors in relative entropy
    epsilon = 1e-10
    P = np.clip(P, epsilon, 1.0)
    Q = np.clip(Q, epsilon, 1.0)
    
    # Re-normalize to ensure they sum to exactly 1.0 after clipping
    P = P / np.sum(P, axis=1, keepdims=True)
    Q = Q / np.sum(Q, axis=1, keepdims=True)
    
    # 3. Calculate Jensen-Shannon Divergence
    # M is the midpoint distribution
    M = 0.5 * (P + Q)
    
    # rel_entr(x, y) computes x * log(x / y). We sum across the features (axis=1).
    kl_pm = np.sum(rel_entr(P, M), axis=1)
    kl_qm = np.sum(rel_entr(Q, M), axis=1)
    
    js_divergences = 0.5 * kl_pm + 0.5 * kl_qm
    
    # 4. Return the average JS Divergence over the forecast horizon m
    return np.mean(js_divergences)

def rolling_window_cv(X, Y, kappa, forecaster):
    T, target_dim = Y.shape[0], Y.shape[1]
    train_size = int(np.floor(T * kappa))
    has_exo = X.shape[1] > target_dim
    results = []
    
    for m in range(T - train_size, 0, -1):
        train_start, train_end = T - train_size - m, T - m
        X_train, Y_train = X[train_start:train_end], Y[train_start:train_end]
        Y_test = Y[train_end : train_end + m]
        future_exo = X[train_end : train_end + m, target_dim:] if has_exo else None
        
        forecaster.fit(X_train, Y_train, val_ratio=0.2)
        seed_seq = X_train[-forecaster.seq_length:]
        Y_pred = forecaster.predict(seed_seq, steps_ahead=m, future_exo=future_exo)
        
        results.append({"m": m, "Sphere_Divergence": calculate_spherical_errors(Y_pred, Y_test)})
    return pd.DataFrame(results)

def rolling_window_cv_JSD(X, Y, kappa, forecaster):
    T, target_dim = Y.shape[0], Y.shape[1]
    train_size = int(np.floor(T * kappa))
    has_exo = X.shape[1] > target_dim
    results = []
    
    for m in range(T - train_size, 0, -1):
        train_start, train_end = T - train_size - m, T - m
        X_train, Y_train = X[train_start:train_end], Y[train_start:train_end]
        Y_test = Y[train_end : train_end + m]
        future_exo = X[train_end : train_end + m, target_dim:] if has_exo else None
        
        forecaster.fit(X_train, Y_train, val_ratio=0.2)
        seed_seq = X_train[-forecaster.seq_length:]
        Y_pred = forecaster.predict(seed_seq, steps_ahead=m, future_exo=future_exo)
        
        results.append({"m": m, "JSD_Divergence": calculate_js_divergence(Y_pred, Y_test)})
    return pd.DataFrame(results)



 ### Real data - U.S. energy generation composition

In [ ]:
df = pd.read_csv('energy.csv', header=None)
Y_raw = df.values
print("Loaded csv")


Y_target = Y_raw / np.linalg.norm(Y_raw, axis=1, keepdims=True)
T, target_features = Y_target.shape


X_base = Y_target.copy()

period = 12 
time_steps = np.arange(T)
sin_time = np.sin(2 * np.pi * time_steps / period).reshape(-1, 1)
cos_time = np.cos(2 * np.pi * time_steps / period).reshape(-1, 1)

X_periodic = np.hstack((Y_target, sin_time, cos_time))

print(f"Target Data (Y) Shape: {Y_target.shape}")
print(f"Base Input (X_base) Shape: {X_base.shape}")
print(f"Periodic Input (X_periodic) Shape: {X_periodic.shape}")


target_dim = Y_target.shape[1]     
base_input_dim = X_base.shape[1]    
periodic_input_dim = X_periodic.shape[1] 
kappa = 0.892
seq_len = 20 

print("--- Training Base Model ---")

base_forecaster = SphericalForecaster(
                    SphericalLSTM(
                        input_dim=base_input_dim, 
                        output_dim=target_dim, 
                        hidden_dim=64
                    ), 
                    seq_length=seq_len, 
                    weight_decay=0.0
                )

df_base = rolling_window_cv(X_base, Y_target, kappa, base_forecaster)

print("\n--- Training Periodic Model ---")             
periodic_forecaster = SphericalForecaster(
                    SphericalLSTM_WithTime(
                        input_dim=periodic_input_dim, 
                        output_dim=target_dim, 
                        hidden_dim=64
                    ), 
                    seq_length=seq_len, 
                    weight_decay=1e-3 
                )


df_periodic = rolling_window_cv(X_periodic, Y_target, kappa, periodic_forecaster)

error_comparison_df = pd.merge(
    df_base, 
    df_periodic, 
    on='m', 
    suffixes=('_Base', '_Periodic')
)

error_comparison_df.columns = [
    'Forecast Horizon (m)', 
    'Base LSTM Error (rad)', 
    'Periodic LSTM Error (rad)'
]


error_comparison_df = error_comparison_df.sort_values('Forecast Horizon (m)')

df_base = df_base.sort_values('m')
df_periodic = df_periodic.sort_values('m')


base_errors = df_base['Sphere_Divergence'].tolist()
periodic_errors = df_periodic['Sphere_Divergence'].tolist()


m_columns = [f"m={int(m)}" for m in df_base['m'].tolist()]


csv_df = pd.DataFrame(
    [base_errors, periodic_errors], 
    columns=m_columns,
    index=["Standard LSTM", "Periodic LSTM"]
)

filename = "energy_LSTM.csv"
csv_df.to_csv(filename, header=True, index=True)

print(f"Successfully saved prediction errors to '{filename}'")



print("=== Mean Geodesic Distance [Dist(m)] Prediction Errors ===\n")
print(error_comparison_df.to_string(index=False, float_format="%.6f"))

# --- Plotting the Comparison ---
plt.figure(figsize=(10, 6))

plt.plot(df_base['m'], df_base['Sphere_Divergence'], marker='o', color='#d62728', linestyle='-', label="Base LSTM")
plt.plot(df_periodic['m'], df_periodic['Sphere_Divergence'], marker='s', color='#1f77b4', linestyle='-', label="Periodic LSTM")

plt.title(r'Prediction Error ${\rm Dist}(m)$: Base vs Periodic Model', fontsize=14)
plt.xlabel('Forecast Horizon (m)', fontsize=12)
plt.ylabel(r'Mean Geodesic Distance (Radians)', fontsize=12)
plt.xticks(df_base['m'])
plt.gca().invert_xaxis() 
plt.legend(fontsize=12)

plt.tight_layout()
plt.show()

### Real data - New York City Citi bike sharing system

In [ ]:

df = pd.read_csv('trip.csv', header=None)
Y_raw = df.values
print("Loaded csv")


Y_target = Y_raw / np.linalg.norm(Y_raw, axis=1, keepdims=True)
T, target_features = Y_target.shape


X_base = Y_target.copy()


period = 7 
time_steps = np.arange(T)
sin_time = np.sin(2 * np.pi * time_steps / period).reshape(-1, 1)
cos_time = np.cos(2 * np.pi * time_steps / period).reshape(-1, 1)

X_periodic = np.hstack((Y_target, sin_time, cos_time))

print(f"Target Data (Y) Shape: {Y_target.shape}")
print(f"Base Input (X_base) Shape: {X_base.shape}")
print(f"Periodic Input (X_periodic) Shape: {X_periodic.shape}")


target_dim = Y_target.shape[1]      
base_input_dim = X_base.shape[1]    
periodic_input_dim = X_periodic.shape[1] 

kappa = 0.86
seq_len = 20 

print("--- Training Base Model ---")
base_forecaster = SphericalForecaster(
                    SphericalLSTM(
                        input_dim=base_input_dim, 
                        output_dim=target_dim, 
                        hidden_dim=64
                    ), 
                    seq_length=seq_len, 
                    weight_decay=0.0
                )

df_base = rolling_window_cv_JSD(X_base, Y_target, kappa, base_forecaster)

print("\n--- Training Periodic Model ---")             
# Initialize Periodic Model automatically
periodic_forecaster = SphericalForecaster(
                    SphericalLSTM_WithTime(
                        input_dim=periodic_input_dim, 
                        output_dim=target_dim, 
                        hidden_dim=64
                    ), 
                    seq_length=seq_len, 
                    weight_decay=1e-3 
                )


df_periodic = rolling_window_cv_JSD(X_periodic, Y_target, kappa, periodic_forecaster)
error_comparison_df = pd.merge(
    df_base, 
    df_periodic, 
    on='m', 
    suffixes=('_Base', '_Periodic')
)


error_comparison_df.columns = [
    'Forecast Horizon (m)', 
    'Base LSTM Error (rad)', 
    'Periodic LSTM Error (rad)'
]

error_comparison_df = error_comparison_df.sort_values('Forecast Horizon (m)')

df_base = df_base.sort_values('m')
df_periodic = df_periodic.sort_values('m')


base_errors = df_base['JSD_Divergence'].tolist()
periodic_errors = df_periodic['JSD_Divergence'].tolist()


m_columns = [f"m={int(m)}" for m in df_base['m'].tolist()]


csv_df = pd.DataFrame(
    [base_errors, periodic_errors], 
    columns=m_columns,
    index=["Standard LSTM", "Periodic LSTM"]
)


filename = "trip_LSTM.csv"
csv_df.to_csv(filename, header=True, index=True)

print(f"Successfully saved prediction errors to '{filename}'")



print("=== Mean Geodesic Distance [Dist(m)] Prediction Errors ===\n")
print(error_comparison_df.to_string(index=False, float_format="%.6f"))

# --- Plotting the Comparison ---
plt.figure(figsize=(10, 6))

plt.plot(df_base['m'], df_base['JSD_Divergence'], marker='o', color='#d62728', linestyle='-', label="Base LSTM")
plt.plot(df_periodic['m'], df_periodic['JSD_Divergence'], marker='s', color='#1f77b4', linestyle='-', label="Periodic LSTM")

plt.title(r'Prediction Error ${\rm Dist}(m)$: Base vs Periodic Model', fontsize=14)
plt.xlabel('Forecast Horizon (m)', fontsize=12)
plt.ylabel(r'Mean Geodesic Distance (Radians)', fontsize=12)
plt.xticks(df_base['m'])
plt.gca().invert_xaxis() 
plt.legend(fontsize=12)

plt.tight_layout()
plt.show()